In [ ]:
# %pip restarts Python, keep before %run

In [ ]:
%pip install -q geopandas shapely pyproj

In [ ]:
%run ../globalvariables

In [ ]:
%run ../lakehousefunction

In [ ]:
DATASET = "distritos"
NOTEBOOK = "bronze/import_distritos"

df_origins = pd.read_csv(ORIGIN_PATH)
df_datasets = pd.read_csv(DATASET_PATH)

row = df_datasets[df_datasets["dataset_destino"] == DATASET].iloc[0]
origin_row = df_origins[df_origins["id"] == row["origin"]].iloc[0]
CKAN_BASE = origin_row["endpoint"]

print(f"{DATASET}: origin={row['dataset_origin']} partition={row['partition']}")

In [ ]:
try:
    probe = fetch_with_retry(lambda: fetch_resources(row["dataset_origin"], CKAN_BASE))
    print(f"CKAN reachable - {len(probe)} resources")
except Exception as e:
    log_errors([error_record(NOTEBOOK, e)])
    print(f"connection failed: {e}")
    dbutils.notebook.exit(1)

In [ ]:
errors = []
ingested = []

resources = fetch_with_retry(lambda: fetch_resources(row["dataset_origin"], CKAN_BASE))
mode = "overwrite"

for key in partition_keys(row["partition"], None, None):
    try:
        resource = pick_resource(resources, row["partition"], key, row["file_type"], row["select_by"])
        df = read_resource(resource, row["file_type"], DATASET, key)
        df = sanitize_column_names(df)
        df = apply_schema(df, DATASET)
        df = add_ingestion_metadata(
            df, source_system="CKAN", source_file=resource["url"],
            partition_key=key or "latest",
        )
        rows = df.count()

        if not write_bronze(df, DATASET, mode=mode):
            raise Exception("write_bronze returned False")

        mode = "append"
        ingested.append({
            "timestamp": datetime.now().isoformat(), "notebook_name": NOTEBOOK,
            "dataset_destino": DATASET, "partition_key": key or "latest",
            "rows_written": rows, "status": "SUCCESS",
        })
        print(f"ok {DATASET} {key or ''} ({rows} rows)")
    except ValueError as e:
        errors.append(error_record(NOTEBOOK, e, status="SKIPPED"))
        print(f"skip {DATASET} {key or ''}: not published yet - {e}")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail {DATASET} {key or ''}: {type(e).__name__}: {e}")

In [ ]:
print(f"{DATASET}: {len(ingested)} partitions ok, {len(errors)} failed")
log_ingestion(ingested)
log_errors(errors)